# 5.10 用來選模型的分數能當最後成績嗎？


兩設定在驗證題得0.7與0.8，我們選0.8那個。這份分數已參與決策，不能再當定案方法在完全獨立考卷上的成績；反覆用模擬試卷挑讀書方法，也會逐漸迎合那份卷子。

訓練資料更新權重，驗證資料選設定，測試資料等方法定案才計分。區別是用途，不只是檔名；下面只執行選設定這件事。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
candidates = [
    {"lr": 0.001, "validation": 0.7},
    {"lr": 0.01, "validation": 0.8},
]
selected = candidates[0]
for candidate in candidates[1:]:
    if candidate["validation"] > selected["validation"]:
        selected = candidate
print("選擇的設定", selected)
print("test仍封存，尚未計算最終成績")

選擇的設定 {'lr': 0.01, 'validation': 0.8}
test仍封存，尚未計算最終成績


輸入兩份假設實驗記錄，lr是學習率、validation是驗證成功比例；這些分數為講解選擇規則而指定，並不是模型測量。先拿第一份當目前最佳，迴圈比較後續候選。`if` 只在條件成立時執行縮排行；0.8大於0.7，所以更新selected。輸出選lr0.01、validation0.8，第二行明確說test還沒評，程式沒有憑空捏造test值。

一次選擇也會受隨機波動影響，候選越多，越容易恰好挑到對這份驗證樣本特別幸運的結果。這個「挑最高」的偏差不能靠只報最優數字消除。應保留各候選記錄、選擇規則與嘗試數量，最終在同一份封存test上報告定案方法，再依 [5.15](https://birdhackor.github.io/tiny-perceptron-vlm/5.15.html) 看不同種子波動。

候選越多，越容易挑到特別適合驗證樣本的一次。保存試過哪些設定、用何指標選；看完test再調參，這份test也加入開發循環。分組應先於切窗口，同故事前半教過、後半評，不宜當獨立新故事；自己的留出成績也不等於官方測試成績。

練習在candidates加一份 `{"lr": 0.1, "validation": 0.6}`，先預測選擇仍為0.01，再執行核對。然後把新增記錄的validation改成0.9，選擇會改，但test行仍應保持「尚未計算」。程式挑到更高驗證數，不等於它已經提供更好的獨立成績。

<details>
<summary>補充：實作約定與原始紀錄</summary>

實際操作還要分清「上游檔名」與「這次實驗的用途」。[T.4的TinyStories資料](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)選自上游的train檔，但我們先保留完整故事、自行切出訓練、驗證和最後檢查，再把各篇切成模型放得下的窗口。因此這次沒有拿自己的驗證或最後檢查故事更新權重；它們仍不等於TinyStories官方測試集，也不能代表對完整資料集的成績。中文詩集本來沒有這三份切分，同樣需要先按整首詩分開，不能把一首詩的前半拿去教、後半當成獨立新詩。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
